<a href="https://colab.research.google.com/github/AhmadSyahreza/Katanya-kami-lomba/blob/main/Bismillah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JOINTS x Inspire — MASE Tuning Besar

Notebook eksperimen untuk kompetisi prediksi tiket bioskop D1–D3 → D4–D10.

**Strategi utama**
- Membangun *sliding windows* dari semua rangkaian 10 hari yang lengkap di `train.csv`.
- Mengubah target menjadi rasio `total_ticket / scale`, sehingga MAE pada rasio sama dengan MASE resmi.
- Memvalidasi pada judul film yang tidak masuk training fold, agar lebih mendekati kondisi test yang didominasi film baru.
- Melakukan random hyperparameter search LightGBM dengan early stopping.
- Melatih ulang model terbaik pada seluruh data dan membuat `submission_tuning.csv`.

> Jalankan di Google Colab. Notebook ini mengasumsikan file kompetisi ada di folder Google Drive `/content/drive/MyDrive/Lomba Data/`. Sesuaikan `BASE` jika lokasi foldermu berbeda.
>
> Target MASE 0.33 adalah sasaran eksperimen, bukan jaminan. Validasi lokal tidak bisa memastikan skor private leaderboard.

## 1. Setup dan reproducibility

Notebook ini memakai seed tetap. Tuning 18 konfigurasi dapat memakan waktu; ubah `N_TRIALS` jika runtime Colab terbatas.

In [1]:
# Jika package belum tersedia, jalankan cell ini.
%pip -q install lightgbm

import os, gc, json, random, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import GroupShuffleSplit, ParameterSampler
from sklearn.metrics import mean_absolute_error

warnings.filterwarnings("ignore")

SEED = 2026
np.random.seed(SEED)
random.seed(SEED)

N_TRIALS = 18
EARLY_STOPPING_ROUNDS = 100
MAX_ESTIMATORS = 1800

print("LightGBM:", lgb.__version__)
print("Seed:", SEED)

LightGBM: 4.6.0
Seed: 2026


## 2. Load data

File yang dipakai: `train.csv`, `test_history.csv`, `test.csv`, `movies.csv`, `ticket_prices.csv`, `holidays.csv`, dan `sample_submission.csv`.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/Lomba Data/"

def read_csv_checked(filename, parse_dates=None):
    path = os.path.join(BASE, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Tidak menemukan {path}. Periksa BASE dan nama file di Google Drive."
        )
    return pd.read_csv(path, parse_dates=parse_dates)

train = read_csv_checked("train.csv", parse_dates=["date_show"])
test_history = read_csv_checked("test_history.csv", parse_dates=["date_show"])
test = read_csv_checked("test.csv", parse_dates=["date_show"])
movies = read_csv_checked("movies.csv")
ticket_prices = read_csv_checked("ticket_prices.csv")
holidays = read_csv_checked("holidays.csv", parse_dates=["date"])
sample_sub = read_csv_checked("sample_submission.csv")

KEYS = ["movie_title", "cinema_ids"]
TARGET_COL = "total_ticket"
TARGET_DAYS = list(range(4, 11))

for df in [train, test_history, test]:
    df["date_show"] = pd.to_datetime(df["date_show"], errors="coerce")

print("train       :", train.shape, train.columns.tolist())
print("test_history:", test_history.shape, test_history.columns.tolist())
print("test        :", test.shape, test.columns.tolist())
print("movies      :", movies.shape)
print("prices      :", ticket_prices.shape)
print("holidays    :", holidays.shape)
print("submission  :", sample_sub.shape)

display(train.head())
display(test.head())

Mounted at /content/drive
train       : (138959, 7) ['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']
test_history: (32323, 7) ['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']
test        : (72611, 5) ['id', 'movie_title', 'cinema_ids', 'city_name', 'date_show']
movies      : (397, 7)
prices      : (207, 3)
holidays    : (366, 4)
submission  : (72611, 2)


,date_show,cinema_ids,city_name,movie_title,total_ticket,occupation_rate,total_show
0,2025-04-01,015555fbcd8faef6b55a854605df0a87,SURABAYA,A WORKING MAN,487,31.07,18
1,2025-04-01,015555fbcd8faef6b55a854605df0a87,SURABAYA,JUMBO,649,23.69,19
2,2025-04-01,015555fbcd8faef6b55a854605df0a87,SURABAYA,KOMANG,389,9.55,22
3,2025-04-01,015555fbcd8faef6b55a854605df0a87,SURABAYA,NE ZHA 2,101,70.03,3
4,2025-04-01,015555fbcd8faef6b55a854605df0a87,SURABAYA,NORMA: ANTARA MERTUA DAN MENANTU,411,17.76,13


,id,movie_title,cinema_ids,city_name,date_show
0,1,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-08
1,2,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-09
2,3,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-10
3,4,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-11
4,5,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-12


## 3. Validasi skema dan metric resmi

Skala per pasangan film–klaster adalah rata-rata tiket D1–D3, dengan minimum 1. Untuk target yang dinormalisasi sebagai `y_ratio = total_ticket / scale`, maka:

`MAE(y_ratio, pred_ratio) = mean(abs(y_true - y_pred) / scale) = MASE`.

Ini membuat tuning langsung mengoptimalkan metric yang dijelaskan panitia.

In [3]:
required_train = set(KEYS + ["city_name", "date_show", "total_ticket",
                             "occupation_rate", "total_show"])
required_test = set(KEYS + ["city_name", "date_show", "id"])
required_hist = set(KEYS + ["city_name", "date_show", "total_ticket",
                            "occupation_rate", "total_show"])

missing = required_train - set(train.columns)
if missing:
    raise ValueError(f"Kolom train.csv tidak lengkap: {sorted(missing)}")
missing = required_test - set(test.columns)
if missing:
    raise ValueError(f"Kolom test.csv tidak lengkap: {sorted(missing)}")
missing = required_hist - set(test_history.columns)
if missing:
    raise ValueError(f"Kolom test_history.csv tidak lengkap: {sorted(missing)}")

def official_mase(y_true, y_pred, scale):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    scale = np.maximum(np.asarray(scale, dtype=float), 1.0)
    return float(np.mean(np.abs(y_true - y_pred) / scale))

print("Train date range:", train.date_show.min(), "to", train.date_show.max())
print("Test date range :", test.date_show.min(), "to", test.date_show.max())
print("Train movies:", train.movie_title.nunique())
print("Test movies :", test.movie_title.nunique())
print("Movie overlap:", len(set(train.movie_title) & set(test.movie_title)))
print("Test rows per pair (counts):")
display(test.groupby(KEYS).size().value_counts().sort_index().rename("number_of_pairs").to_frame())

Train date range: 2025-04-01 00:00:00 to 2025-09-30 00:00:00
Test date range : 2025-10-04 00:00:00 to 2026-03-27 00:00:00
Train movies: 237
Test movies : 160
Movie overlap: 10
Test rows per pair (counts):


,number_of_pairs
7,10373


## 4. Bangun semua sliding windows yang valid

Versi sebelumnya hanya mengambil `g.iloc[:10]` per pasangan. Di sini kita memakai setiap window 10 hari yang berurutan:
- tiga baris pertama menjadi fitur observasi;
- tujuh baris berikutnya menjadi target;
- window dilewati jika tanggal tidak berurutan tepat satu hari.

Window yang tumpang tindih tetap dikelompokkan berdasarkan judul film saat validasi agar film yang sama tidak bocor antara training dan validation.

In [4]:
OBS_COLS = ["total_ticket", "occupation_rate", "total_show"]

def build_sliding_windows(df):
    rows = []
    df = df.dropna(subset=KEYS + ["city_name", "date_show"]).copy()
    df = df.sort_values(KEYS + ["date_show"])

    for (movie, cinema), g in df.groupby(KEYS, sort=False, observed=True):
        g = g.sort_values("date_show").reset_index(drop=True)
        if len(g) < 10:
            continue

        dates = pd.to_datetime(g["date_show"]).to_numpy(dtype="datetime64[ns]")
        tickets = pd.to_numeric(g["total_ticket"], errors="coerce").to_numpy(dtype=float)
        occ = pd.to_numeric(g["occupation_rate"], errors="coerce").to_numpy(dtype=float)
        shows = pd.to_numeric(g["total_show"], errors="coerce").to_numpy(dtype=float)
        city = g["city_name"].iloc[0]

        for start in range(len(g) - 9):
            idx = np.arange(start, start + 10)
            window_dates = dates[idx]
            if not np.all(np.diff(window_dates) == np.timedelta64(1, "D")):
                continue
            if not np.isfinite(tickets[idx]).all():
                continue

            r = {
                "movie_title": movie,
                "cinema_ids": cinema,
                "city_name": city,
                "date_D1": pd.Timestamp(window_dates[0]),
            }
            for j in range(3):
                r[f"d{j+1}"] = tickets[start+j]
                r[f"occ_{j+1}"] = occ[start+j] if np.isfinite(occ[start+j]) else np.nan
                r[f"show_{j+1}"] = shows[start+j] if np.isfinite(shows[start+j]) else np.nan
            r["scale"] = max(float(np.mean(tickets[start:start+3])), 1.0)

            for h in range(4, 11):
                r[f"target_{h}"] = tickets[start + h - 1]
                r[f"date_D{h}"] = pd.Timestamp(window_dates[h - 1])
            rows.append(r)

    result = pd.DataFrame(rows)
    if result.empty:
        raise ValueError("Tidak ada window 10 hari berurutan. Periksa struktur train.csv.")
    return result

windows = build_sliding_windows(train)
print("Valid windows:", windows.shape)
print("Unique movie titles:", windows.movie_title.nunique())
display(windows.head())

Valid windows: (48466, 28)
Unique movie titles: 149


,movie_title,cinema_ids,city_name,date_D1,d1,occ_1,show_1,d2,occ_2,show_2,...,target_6,date_D6,target_7,date_D7,target_8,date_D8,target_9,date_D9,target_10,date_D10
0,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-18,1376.0,39.38,23.0,1033.0,23.87,23.0,...,804.0,2025-06-23,576.0,2025-06-24,548.0,2025-06-25,484.0,2025-06-26,834.0,2025-06-27
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-19,1033.0,23.87,23.0,946.0,22.75,23.0,...,576.0,2025-06-24,548.0,2025-06-25,484.0,2025-06-26,834.0,2025-06-27,649.0,2025-06-28
2,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-20,946.0,22.75,23.0,1455.0,34.35,23.0,...,548.0,2025-06-25,484.0,2025-06-26,834.0,2025-06-27,649.0,2025-06-28,528.0,2025-06-29
3,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-21,1455.0,34.35,23.0,1103.0,24.50,24.0,...,484.0,2025-06-26,834.0,2025-06-27,649.0,2025-06-28,528.0,2025-06-29,330.0,2025-06-30
4,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-22,1103.0,24.50,24.0,804.0,20.48,24.0,...,834.0,2025-06-27,649.0,2025-06-28,528.0,2025-06-29,330.0,2025-06-30,361.0,2025-07-01


## 5. Ubah ke format long dan buat fitur

Satu baris model mewakili satu pasangan-window-hari target. Fitur kalender hanya menggunakan tanggal target yang diketahui dari jadwal kompetisi. Tidak ada tiket aktual D4–D10 yang dipakai sebagai fitur.

In [5]:
def expand_to_long(wide):
    parts = []
    for h in TARGET_DAYS:
        part = wide.drop(columns=[f"target_{d}" for d in TARGET_DAYS if d != h] +
                         [f"date_D{d}" for d in TARGET_DAYS if d != h],
                         errors="ignore").copy()
        part = part.rename(columns={f"target_{h}": "y", f"date_D{h}": "target_date"})
        part["horizon"] = h
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

train_long = expand_to_long(windows)
train_long["y"] = pd.to_numeric(train_long["y"], errors="coerce")
train_long["y_ratio"] = train_long["y"] / train_long["scale"]
train_long = train_long.replace([np.inf, -np.inf], np.nan)
train_long = train_long.dropna(subset=["y", "y_ratio", "target_date"]).reset_index(drop=True)

print("Long-format rows:", train_long.shape)
print("MASE scale summary:")
display(train_long["scale"].describe())

Long-format rows: (339262, 18)
MASE scale summary:


,scale
count,339262.000000
mean,679.128606
std,1016.998756
min,9.666667
25%,217.000000
50%,389.166667
75%,743.333333
max,26989.000000


## 6. Metadata, harga tiket, dan kalender

Metadata yang dipakai hanya informasi eksternal yang tersedia sebagai file kompetisi: genre, age rating, harga rata-rata per kota, serta tipe hari/libur. Jika nama kolom file metadata berbeda, cell ini akan memberi tahu kolom yang tersedia.

In [6]:
# Movie metadata
movie_cols = [c for c in ["original_title", "genre", "age_rating"] if c in movies.columns]
if "original_title" in movie_cols:
    movies_meta = movies[movie_cols].drop_duplicates("original_title").copy()
    movies_meta = movies_meta.rename(columns={"original_title": "movie_title"})
else:
    movies_meta = pd.DataFrame(columns=["movie_title", "genre", "age_rating"])

if "genre" not in movies_meta.columns:
    movies_meta["genre"] = "Unknown"
if "age_rating" not in movies_meta.columns:
    movies_meta["age_rating"] = "Unknown"

# City-level ticket price summary
if {"city_name", "ceil"}.issubset(ticket_prices.columns):
    price_meta = (
        ticket_prices.groupby("city_name", as_index=False)["ceil"]
        .mean().rename(columns={"ceil": "ticket_price_mean"})
    )
elif {"city_name", "ticket_price"}.issubset(ticket_prices.columns):
    price_meta = (
        ticket_prices.groupby("city_name", as_index=False)["ticket_price"]
        .mean().rename(columns={"ticket_price": "ticket_price_mean"})
    )
else:
    price_meta = pd.DataFrame(columns=["city_name", "ticket_price_mean"])

holiday_cols = ["date"] + [c for c in ["day_tipe", "holiday_tipe"] if c in holidays.columns]
holiday_meta = holidays[holiday_cols].drop_duplicates("date").copy()
holiday_meta["date"] = pd.to_datetime(holiday_meta["date"], errors="coerce")

def add_calendar_metadata(df):
    out = df.copy()
    out["target_date"] = pd.to_datetime(out["target_date"], errors="coerce")
    out = out.merge(movies_meta, on="movie_title", how="left")
    out = out.merge(price_meta, on="city_name", how="left")
    out = out.merge(holiday_meta, left_on="target_date", right_on="date", how="left")
    if "date" in out.columns:
        out = out.drop(columns=["date"])

    out["genre"] = out["genre"].fillna("Unknown").astype(str)
    out["genre_main"] = out["genre"].str.split(",").str[0].str.strip().replace("", "Unknown")
    out["age_rating"] = out["age_rating"].fillna("Unknown").astype(str)
    out["day_tipe"] = out["day_tipe"].fillna("weekday").astype(str) if "day_tipe" in out.columns else "weekday"
    out["holiday_tipe"] = out["holiday_tipe"].fillna("normal").astype(str) if "holiday_tipe" in out.columns else "normal"
    out["ticket_price_mean"] = pd.to_numeric(out["ticket_price_mean"], errors="coerce") if "ticket_price_mean" in out.columns else np.nan

    dt = out["target_date"]
    out["target_dow"] = dt.dt.dayofweek
    out["target_day"] = dt.dt.day
    out["target_month"] = dt.dt.month
    out["target_dayofyear"] = dt.dt.dayofyear
    out["target_weekofyear"] = dt.dt.isocalendar().week.astype("float")
    out["target_is_weekend"] = (dt.dt.dayofweek >= 5).astype(int)
    out["target_is_month_start"] = dt.dt.is_month_start.astype(int)
    out["target_is_month_end"] = dt.dt.is_month_end.astype(int)
    out["target_is_holiday"] = (out["holiday_tipe"].str.lower() == "holiday").astype(int)

    # History features
    out["hist_mean"] = out[["d1", "d2", "d3"]].mean(axis=1)
    out["hist_std"] = out[["d1", "d2", "d3"]].std(axis=1).fillna(0)
    out["hist_min"] = out[["d1", "d2", "d3"]].min(axis=1)
    out["hist_max"] = out[["d1", "d2", "d3"]].max(axis=1)
    out["hist_median"] = out[["d1", "d2", "d3"]].median(axis=1)
    out["hist_range"] = out["hist_max"] - out["hist_min"]
    out["trend_13"] = (out["d3"] - out["d1"]) / (out["scale"] + 1)
    out["trend_12"] = (out["d2"] - out["d1"]) / (out["scale"] + 1)
    out["trend_23"] = (out["d3"] - out["d2"]) / (out["scale"] + 1)
    out["ratio_21"] = (out["d2"] + 1) / (out["d1"] + 1)
    out["ratio_31"] = (out["d3"] + 1) / (out["d1"] + 1)
    out["ratio_32"] = (out["d3"] + 1) / (out["d2"] + 1)
    out["log_scale"] = np.log1p(out["scale"])
    out["occ_mean"] = out[["occ_1", "occ_2", "occ_3"]].mean(axis=1)
    out["occ_std"] = out[["occ_1", "occ_2", "occ_3"]].std(axis=1).fillna(0)
    out["occ_trend"] = out["occ_3"] - out["occ_1"]
    out["show_mean"] = out[["show_1", "show_2", "show_3"]].mean(axis=1)
    out["show_trend"] = out["show_3"] - out["show_1"]
    out["tickets_per_show"] = out["hist_mean"] / (out["show_mean"] + 1)
    out["tickets_per_occupancy"] = out["hist_mean"] / (out["occ_mean"].abs() + 0.01)
    out["horizon_float"] = out["horizon"].astype(float)
    out["horizon_sq"] = out["horizon"].astype(float) ** 2

    return out

train_long = add_calendar_metadata(train_long)
print("Feature table:", train_long.shape)
display(train_long.head())

Feature table: (339262, 55)


,movie_title,cinema_ids,city_name,date_D1,d1,occ_1,show_1,d2,occ_2,show_2,...,log_scale,occ_mean,occ_std,occ_trend,show_mean,show_trend,tickets_per_show,tickets_per_occupancy,horizon_float,horizon_sq
0,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-18,1376.0,39.38,23.0,1033.0,23.87,23.0,...,7.020489,28.666667,9.294904,-16.63,23.000000,0.0,46.597222,38.998024,4.0,16.0
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-19,1033.0,23.87,23.0,946.0,22.75,23.0,...,7.043742,26.990000,6.398500,10.48,23.000000,0.0,47.694444,42.395062,4.0,16.0
2,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-20,946.0,22.75,23.0,1455.0,34.35,23.0,...,7.063904,27.200000,6.253599,1.75,23.333333,1.0,48.000000,42.925395,4.0,16.0
3,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-21,1455.0,34.35,23.0,1103.0,24.50,24.0,...,7.022571,26.443333,7.136290,-13.87,23.666667,1.0,45.432432,42.363911,4.0,16.0
4,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-06-22,1103.0,24.50,24.0,804.0,20.48,24.0,...,6.719818,20.016667,4.732043,-9.43,23.666667,-1.0,33.554054,41.328229,4.0,16.0


## 7. Group-based validation: film validation tidak muncul di training fold

Validation split dibuat berdasarkan `movie_title`, bukan baris acak. Karena satu judul film memiliki banyak window dan tujuh horizon, pemisahan baris acak akan membuat pola film yang sama masuk ke kedua sisi.

Model utama memprediksi rasio tiket. Metrik validation dihitung ulang sebagai MASE resmi untuk memastikan kesesuaiannya.

In [7]:
groups = train_long["movie_title"].astype(str).values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=SEED)
tr_idx, va_idx = next(splitter.split(train_long, groups=groups))

tr_df = train_long.iloc[tr_idx].reset_index(drop=True)
va_df = train_long.iloc[va_idx].reset_index(drop=True)

print("Train rows:", len(tr_df), "| movies:", tr_df.movie_title.nunique())
print("Valid rows:", len(va_df), "| movies:", va_df.movie_title.nunique())
print("Movie overlap:", len(set(tr_df.movie_title) & set(va_df.movie_title)))

# Baseline: median target ratio per horizon learned from training fold only
global_ratio = tr_df.groupby("horizon")["y_ratio"].median().to_dict()
baseline_ratio = va_df["horizon"].map(global_ratio).to_numpy()
baseline_pred = np.maximum(0, baseline_ratio) * va_df["scale"].to_numpy()
baseline_score = official_mase(va_df["y"].to_numpy(), baseline_pred, va_df["scale"].to_numpy())
print(f"Baseline median-ratio MASE: {baseline_score:.5f}")

Train rows: 246687 | movies: 116
Valid rows: 92575 | movies: 33
Movie overlap: 0
Baseline median-ratio MASE: 0.33021


## 8. Feature matrix dan categorical handling

`movie_title`, `cinema_ids`, dan kategori metadata diperlakukan sebagai categorical features. Judul film validation yang belum pernah terlihat akan menjadi kategori missing, sehingga model tidak dapat menghafal target film validation.

In [8]:
CAT_COLS = [
    "movie_title", "cinema_ids", "city_name",
    "genre_main", "age_rating", "day_tipe", "holiday_tipe"
]

DROP_COLS = {
    "y", "y_ratio", "target_date", "date_D1",
    "genre", "total_ticket"
}

FEATURE_COLS = [
    c for c in train_long.columns
    if c not in DROP_COLS and c not in [f"target_{h}" for h in TARGET_DAYS]
    and c not in [f"date_D{h}" for h in TARGET_DAYS]
]

# Keep only known feature columns, excluding accidental object columns not designated categorical.
for c in CAT_COLS:
    if c not in FEATURE_COLS:
        FEATURE_COLS.append(c)

FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_long.columns]

print("Number of features:", len(FEATURE_COLS))
print("Categorical features:", [c for c in CAT_COLS if c in FEATURE_COLS])
print(FEATURE_COLS)

def make_lgb_matrices(train_part, valid_part, feature_cols):
    Xtr = train_part[feature_cols].copy()
    Xva = valid_part[feature_cols].copy()
    cats = [c for c in CAT_COLS if c in feature_cols]

    for c in cats:
        Xtr[c] = Xtr[c].fillna("Unknown").astype(str)
        Xva[c] = Xva[c].fillna("Unknown").astype(str)
        categories = pd.Index(Xtr[c].unique())
        Xtr[c] = pd.Categorical(Xtr[c], categories=categories)
        Xva[c] = pd.Categorical(Xva[c], categories=categories)

    for c in Xtr.columns:
        if c not in cats:
            Xtr[c] = pd.to_numeric(Xtr[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            Xva[c] = pd.to_numeric(Xva[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            med = Xtr[c].median()
            if pd.isna(med):
                med = 0
            Xtr[c] = Xtr[c].fillna(med)
            Xva[c] = Xva[c].fillna(med)

    return Xtr, Xva, cats

Xtr, Xva, CAT_FEATURES = make_lgb_matrices(tr_df, va_df, FEATURE_COLS)
ytr = tr_df["y_ratio"].astype(float).to_numpy()
yva = va_df["y_ratio"].astype(float).to_numpy()
print("Matrices:", Xtr.shape, Xva.shape)

Number of features: 50
Categorical features: ['movie_title', 'cinema_ids', 'city_name', 'genre_main', 'age_rating', 'day_tipe', 'holiday_tipe']
['movie_title', 'cinema_ids', 'city_name', 'd1', 'occ_1', 'show_1', 'd2', 'occ_2', 'show_2', 'd3', 'occ_3', 'show_3', 'scale', 'horizon', 'age_rating', 'ticket_price_mean', 'day_tipe', 'holiday_tipe', 'genre_main', 'target_dow', 'target_day', 'target_month', 'target_dayofyear', 'target_weekofyear', 'target_is_weekend', 'target_is_month_start', 'target_is_month_end', 'target_is_holiday', 'hist_mean', 'hist_std', 'hist_min', 'hist_max', 'hist_median', 'hist_range', 'trend_13', 'trend_12', 'trend_23', 'ratio_21', 'ratio_31', 'ratio_32', 'log_scale', 'occ_mean', 'occ_std', 'occ_trend', 'show_mean', 'show_trend', 'tickets_per_show', 'tickets_per_occupancy', 'horizon_float', 'horizon_sq']
Matrices: (246687, 50) (92575, 50)


## 9. Hyperparameter tuning besar-besaran

Melakukan random search deterministik dengan 18 konfigurasi. Setiap percobaan menggunakan early stopping pada validation MASE. Ini lebih terukur daripada mengganti parameter secara manual berdasarkan satu submission.

Kalau Colab kehabisan waktu, kurangi `N_TRIALS` menjadi 8–10 dan lanjutkan dengan hasil yang tersedia.

In [9]:
param_space = {
    "learning_rate": [0.015, 0.025, 0.035, 0.05, 0.07],
    "num_leaves": [15, 23, 31, 47, 63, 95],
    "max_depth": [-1, 5, 7, 9, 12],
    "min_child_samples": [20, 40, 70, 100, 150],
    "subsample": [0.75, 0.85, 0.95, 1.0],
    "colsample_bytree": [0.70, 0.80, 0.90, 1.0],
    "reg_alpha": [0.0, 0.05, 0.2, 0.5, 1.0],
    "reg_lambda": [0.1, 0.5, 1.0, 3.0, 8.0],
    "min_split_gain": [0.0, 0.01, 0.05, 0.1],
}

sampled_params = list(ParameterSampler(
    param_space, n_iter=N_TRIALS, random_state=SEED
))

tuning_results = []
best_model = None
best_score = np.inf
best_iteration = None
best_params = None

for trial, params in enumerate(sampled_params, start=1):
    start_time = time.time()
    model = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=MAX_ESTIMATORS,
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        subsample_freq=1,
        **params
    )

    model.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)],
        eval_metric="l1",
        categorical_feature=CAT_FEATURES,
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=0)
        ]
    )

    pred_ratio = np.clip(model.predict(Xva, num_iteration=model.best_iteration_), 0, None)
    pred_tickets = pred_ratio * va_df["scale"].to_numpy()
    score = official_mase(
        va_df["y"].to_numpy(),
        pred_tickets,
        va_df["scale"].to_numpy()
    )
    elapsed = time.time() - start_time
    iteration = int(model.best_iteration_ or MAX_ESTIMATORS)

    row = {
        "trial": trial,
        "validation_mase": score,
        "best_iteration": iteration,
        "seconds": elapsed,
        **params
    }
    tuning_results.append(row)

    if score < best_score:
        best_score = score
        best_model = model
        best_iteration = iteration
        best_params = params.copy()

    print(
        f"[{trial:02d}/{N_TRIALS}] MASE={score:.5f} | "
        f"iter={iteration} | best={best_score:.5f} | {elapsed:.1f}s"
    )

tuning_df = pd.DataFrame(tuning_results).sort_values("validation_mase").reset_index(drop=True)
display(tuning_df.head(10))
print("Best validation MASE:", best_score)
print("Best iteration:", best_iteration)
print("Best params:", best_params)

[01/18] MASE=0.31458 | iter=70 | best=0.31458 | 16.8s
[02/18] MASE=0.31473 | iter=78 | best=0.31458 | 18.1s
[03/18] MASE=0.31485 | iter=201 | best=0.31458 | 23.5s
[04/18] MASE=0.31481 | iter=181 | best=0.31458 | 22.4s
[05/18] MASE=0.31362 | iter=59 | best=0.31362 | 11.9s
[06/18] MASE=0.31259 | iter=180 | best=0.31259 | 22.7s
[07/18] MASE=0.31512 | iter=55 | best=0.31259 | 16.2s
[08/18] MASE=0.31765 | iter=57 | best=0.31259 | 17.3s
[09/18] MASE=0.31801 | iter=24 | best=0.31259 | 14.4s
[10/18] MASE=0.31610 | iter=25 | best=0.31259 | 11.1s
[11/18] MASE=0.31443 | iter=94 | best=0.31259 | 19.2s
[12/18] MASE=0.31531 | iter=197 | best=0.31259 | 26.8s
[13/18] MASE=0.31267 | iter=59 | best=0.31259 | 15.9s
[14/18] MASE=0.31805 | iter=74 | best=0.31259 | 24.6s
[15/18] MASE=0.32051 | iter=53 | best=0.31259 | 18.9s
[16/18] MASE=0.31368 | iter=116 | best=0.31259 | 16.6s
[17/18] MASE=0.31368 | iter=43 | best=0.31259 | 11.8s
[18/18] MASE=0.31519 | iter=41 | best=0.31259 | 11.7s


,trial,validation_mase,best_iteration,seconds,subsample,reg_lambda,reg_alpha,num_leaves,min_split_gain,min_child_samples,max_depth,learning_rate,colsample_bytree
0,6,0.312592,180,22.713321,0.75,0.5,0.50,95,0.00,40,5,0.015,0.7
1,13,0.312674,59,15.943593,0.75,0.1,0.05,47,0.00,70,12,0.025,0.7
2,5,0.313616,59,11.865928,0.85,3.0,0.05,95,0.10,40,5,0.035,0.7
3,16,0.313678,116,16.602243,0.75,1.0,1.00,31,0.00,100,5,0.025,0.8
4,17,0.313682,43,11.753158,1.00,3.0,0.50,23,0.00,150,5,0.070,0.8
5,11,0.314427,94,19.222928,0.95,8.0,1.00,31,0.01,100,9,0.025,0.8
6,1,0.314579,70,16.830156,0.85,8.0,0.20,31,0.00,70,9,0.035,0.8
7,2,0.314728,78,18.131344,0.95,3.0,0.05,47,0.01,100,7,0.015,0.7
8,4,0.314808,181,22.399806,0.85,0.1,0.05,31,0.05,20,5,0.015,0.8
9,3,0.314846,201,23.451110,0.75,0.1,0.00,15,0.01,150,12,0.035,1.0


Best validation MASE: 0.3125917086802106
Best iteration: 180
Best params: {'subsample': 0.75, 'reg_lambda': 0.5, 'reg_alpha': 0.5, 'num_leaves': 95, 'min_split_gain': 0.0, 'min_child_samples': 40, 'max_depth': 5, 'learning_rate': 0.015, 'colsample_bytree': 0.7}


## 10. Analisis error per horizon dan kalibrasi sederhana

Jangan hanya melihat satu angka. Kita evaluasi D4–D10 satu per satu. Kalibrasi global kecil dapat membantu bila model secara konsisten terlalu tinggi/rendah, tetapi faktor kalibrasi dipilih hanya dari validation set.

In [10]:
best_valid_ratio = np.clip(best_model.predict(Xva, num_iteration=best_iteration), 0, None)
best_valid_pred = best_valid_ratio * va_df["scale"].to_numpy()

valid_eval = va_df[["horizon", "y", "scale"]].copy()
valid_eval["pred"] = best_valid_pred
valid_eval["abs_scaled_error"] = np.abs(valid_eval["y"] - valid_eval["pred"]) / valid_eval["scale"]

horizon_report = (
    valid_eval.groupby("horizon")
    .agg(
        MASE=("abs_scaled_error", "mean"),
        actual_mean=("y", "mean"),
        predicted_mean=("pred", "mean"),
        n=("y", "size")
    )
    .reset_index()
)
display(horizon_report)

# Estimate a single multiplicative calibration factor on validation only.
valid_ratio_actual = va_df["y"].to_numpy() / va_df["scale"].to_numpy()
valid_ratio_pred = np.maximum(best_valid_ratio, 1e-8)
alpha_grid = np.linspace(0.75, 1.25, 101)
alpha_scores = []
for alpha in alpha_grid:
    alpha_scores.append(np.mean(np.abs(valid_ratio_actual - alpha * valid_ratio_pred)))
best_alpha = float(alpha_grid[int(np.argmin(alpha_scores))])
print("Best global calibration alpha (validation only):", best_alpha)
print("Uncalibrated validation MASE:", best_score)
print("Calibrated validation MASE:", float(np.min(alpha_scores)))

,horizon,MASE,actual_mean,predicted_mean,n
0,4,0.273633,508.907599,410.603352,13225
1,5,0.314634,467.559471,369.330201,13225
2,6,0.337970,433.727410,326.103818,13225
3,7,0.340966,409.022004,290.063464,13225
4,8,0.319922,384.821248,264.658281,13225
5,9,0.307680,361.262911,240.320408,13225
6,10,0.293337,334.409603,223.760268,13225


Best global calibration alpha (validation only): 1.24
Uncalibrated validation MASE: 0.3125917086802106
Calibrated validation MASE: 0.2869228361302071


## 11. Buat fitur test dari `test_history` D1–D3

Test memiliki tujuh baris target per pasangan. Fitur historis hanya dibangun dari `test_history`; target masa depan tidak dipakai. Horizon ditentukan berdasarkan urutan tanggal pada masing-masing pasangan.

In [12]:

def build_test_pair_features(history):
    rows = []

    history = history.copy()
    history["date_show"] = pd.to_datetime(history["date_show"], errors="coerce")
    history = history.dropna(subset=KEYS + ["date_show"])
    history = history.sort_values(KEYS + ["date_show"])

    for (movie, cinema), g in history.groupby(
        KEYS, sort=False, observed=True
    ):
        g = g.sort_values("date_show").copy()

        # Combine duplicate records for the same date, if any.
        g = g.groupby("date_show", as_index=False).agg({
            "city_name": "first",
            "total_ticket": "sum",
            "occupation_rate": "mean",
            "total_show": "sum"
        })

        # Build a consecutive 3-day history.
        start_date = g["date_show"].min()
        expected_dates = pd.date_range(start_date, periods=3, freq="D")

        g = g.set_index("date_show").reindex(expected_dates)
        g.index.name = "date_show"

        # Missing history dates are treated as zero activity.
        for col in ["total_ticket", "occupation_rate", "total_show"]:
            g[col] = pd.to_numeric(g[col], errors="coerce").fillna(0)

        tickets = g["total_ticket"].to_numpy(dtype=float)
        occ = g["occupation_rate"].to_numpy(dtype=float)
        shows = g["total_show"].to_numpy(dtype=float)

        city_values = history.loc[
            (history["movie_title"] == movie) &
            (history["cinema_ids"] == cinema),
            "city_name"
        ].dropna()

        r = {
            "movie_title": movie,
            "cinema_ids": cinema,
            "city_name": city_values.iloc[0] if len(city_values) else "Unknown",
            "date_D1": pd.Timestamp(expected_dates[0]),
            "scale": max(float(np.mean(tickets)), 1.0),
        }

        for j in range(3):
            r[f"d{j+1}"] = tickets[j]
            r[f"occ_{j+1}"] = occ[j]
            r[f"show_{j+1}"] = shows[j]

        rows.append(r)

    return pd.DataFrame(rows)


test_pair = build_test_pair_features(test_history)

print("Test history pairs:", len(test_pair))
print("Pairs with missing features:", test_pair[
    ["d1", "d2", "d3", "scale"]
].isna().any(axis=1).sum())

display(test_pair.head())

Test history pairs: 11823
Pairs with missing features: 0


,movie_title,cinema_ids,city_name,date_D1,scale,d1,occ_1,show_1,d2,occ_2,show_2,d3,occ_3,show_3
0,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-05,71.333333,58.0,5.15,7.0,97.0,12.36,5.0,59.0,13.52,3.0
1,"13 DAYS, 13 NIGHTS",0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-12-05,158.000000,135.0,13.65,7.0,191.0,27.04,5.0,148.0,20.81,5.0
2,"13 DAYS, 13 NIGHTS",108f9a4d181f12fd1b7e3c9adae38478,TANGERANG,2025-12-05,26.000000,74.0,5.67,8.0,4.0,1.04,2.0,0.0,0.00,0.0
3,"13 DAYS, 13 NIGHTS",14b63ff2b104903ead7d884f25382e57,MALANG,2025-12-05,10.666667,7.0,2.63,2.0,22.0,16.48,1.0,3.0,2.08,1.0
4,"13 DAYS, 13 NIGHTS",15c7dc37db27dd76f38cc419107f7c48,KARAWANG,2025-12-06,24.333333,50.0,16.80,3.0,23.0,11.44,2.0,0.0,0.00,0.0


## 12. Refit model terbaik pada seluruh windows

Setelah hyperparameter dipilih melalui validation, model dilatih ulang pada semua historical windows. Kategori di test yang tidak pernah terlihat saat training akan menjadi missing category, bukan dipetakan ke target statistik.

In [15]:


# Build test features
test_rows = test.copy()
test_rows["date_show"] = pd.to_datetime(
    test_rows["date_show"], errors="coerce"
)

test_pair = build_test_pair_features(test_history)

test_rows = test_rows.merge(
    test_pair,
    on=KEYS,
    how="left",
    suffixes=("", "_history")
)

# Expand each prediction row across horizons D4-D10
test_long = test_rows.copy()


print("Kolom test:")
print(test.columns.tolist())

print("\nContoh test:")
display(test.head(10))

print("\nKolom test_history:")
print(test_history.columns.tolist())

print("\nContoh test_history:")
display(test_history.head(10))

print("\nKolom training:")
print(train.columns.tolist())

# Use the same horizon definition as training:
# D4 = horizon 4, ..., D10 = horizon 10.
# Replace this assignment with the competition's actual
# D4-D10 mapping if it is defined elsewhere in your notebook.


Kolom test:
['id', 'movie_title', 'cinema_ids', 'city_name', 'date_show']

Contoh test:


,id,movie_title,cinema_ids,city_name,date_show
0,1,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-08
1,2,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-09
2,3,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-10
3,4,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-11
4,5,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-12
5,6,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-13
6,7,"13 DAYS, 13 NIGHTS",015555fbcd8faef6b55a854605df0a87,SURABAYA,2025-12-14
7,8,"13 DAYS, 13 NIGHTS",0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-12-08
8,9,"13 DAYS, 13 NIGHTS",0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-12-09
9,10,"13 DAYS, 13 NIGHTS",0191582d75f94a412c0fa12d36bcee3f,SURABAYA,2025-12-10



Kolom test_history:
['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']

Contoh test_history:


,date_show,cinema_ids,city_name,movie_title,total_ticket,occupation_rate,total_show
0,2025-12-05,015555fbcd8faef6b55a854605df0a87,SURABAYA,"13 DAYS, 13 NIGHTS",58,5.15,7
1,2025-12-06,015555fbcd8faef6b55a854605df0a87,SURABAYA,"13 DAYS, 13 NIGHTS",97,12.36,5
2,2025-12-07,015555fbcd8faef6b55a854605df0a87,SURABAYA,"13 DAYS, 13 NIGHTS",59,13.52,3
3,2025-12-05,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,"13 DAYS, 13 NIGHTS",135,13.65,7
4,2025-12-06,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,"13 DAYS, 13 NIGHTS",191,27.04,5
5,2025-12-07,0191582d75f94a412c0fa12d36bcee3f,SURABAYA,"13 DAYS, 13 NIGHTS",148,20.81,5
6,2025-12-05,108f9a4d181f12fd1b7e3c9adae38478,TANGERANG,"13 DAYS, 13 NIGHTS",74,5.67,8
7,2025-12-06,108f9a4d181f12fd1b7e3c9adae38478,TANGERANG,"13 DAYS, 13 NIGHTS",4,1.04,2
8,2025-12-05,14b63ff2b104903ead7d884f25382e57,MALANG,"13 DAYS, 13 NIGHTS",7,2.63,2
9,2025-12-06,14b63ff2b104903ead7d884f25382e57,MALANG,"13 DAYS, 13 NIGHTS",22,16.48,1



Kolom training:
['date_show', 'cinema_ids', 'city_name', 'movie_title', 'total_ticket', 'occupation_rate', 'total_show']


In [17]:

missing = [c for c in FEATURE_COLS if c not in test_long.columns]

print("Jumlah fitur yang dibutuhkan:", len(FEATURE_COLS))
print("Jumlah fitur yang hilang:", len(missing))
print("\nFitur yang hilang:")
print(missing)

print("\nKolom test_long yang tersedia:")
print(test_long.columns.tolist())


Jumlah fitur yang dibutuhkan: 50
Jumlah fitur yang hilang: 37

Fitur yang hilang:
['horizon', 'age_rating', 'ticket_price_mean', 'day_tipe', 'holiday_tipe', 'genre_main', 'target_dow', 'target_day', 'target_month', 'target_dayofyear', 'target_weekofyear', 'target_is_weekend', 'target_is_month_start', 'target_is_month_end', 'target_is_holiday', 'hist_mean', 'hist_std', 'hist_min', 'hist_max', 'hist_median', 'hist_range', 'trend_13', 'trend_12', 'trend_23', 'ratio_21', 'ratio_31', 'ratio_32', 'log_scale', 'occ_mean', 'occ_std', 'occ_trend', 'show_mean', 'show_trend', 'tickets_per_show', 'tickets_per_occupancy', 'horizon_float', 'horizon_sq']

Kolom test_long yang tersedia:
['id', 'movie_title', 'cinema_ids', 'city_name', 'date_show', 'city_name_history', 'date_D1', 'scale', 'd1', 'occ_1', 'show_1', 'd2', 'occ_2', 'show_2', 'd3', 'occ_3', 'show_3']


In [16]:
X_all = train_long[FEATURE_COLS].copy()
X_test = test_long[FEATURE_COLS].copy()
cats_all = [c for c in CAT_COLS if c in FEATURE_COLS]

for c in cats_all:
    X_all[c] = X_all[c].fillna("Unknown").astype(str)
    X_test[c] = X_test[c].fillna("Unknown").astype(str)
    categories = pd.Index(X_all[c].unique())
    X_all[c] = pd.Categorical(X_all[c], categories=categories)
    X_test[c] = pd.Categorical(X_test[c], categories=categories)

for c in X_all.columns:
    if c not in cats_all:
        X_all[c] = pd.to_numeric(X_all[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        X_test[c] = pd.to_numeric(X_test[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        med = X_all[c].median()
        if pd.isna(med):
            med = 0
        X_all[c] = X_all[c].fillna(med)
        X_test[c] = X_test[c].fillna(med)

final_model = lgb.LGBMRegressor(
    objective="regression_l1",
    n_estimators=max(int(best_iteration or 500), 100),
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    subsample_freq=1,
    **best_params
)
final_model.fit(
    X_all,
    train_long["y_ratio"].astype(float),
    categorical_feature=cats_all
)

test_ratio_pred = np.clip(final_model.predict(X_test), 0, None)
# Apply validation-derived global calibration. Set to 1.0 if you prefer no calibration.
USE_CALIBRATION = True
alpha_final = best_alpha if USE_CALIBRATION else 1.0
test_pred = test_ratio_pred * alpha_final * test_long["scale"].to_numpy()
test_pred = np.maximum(test_pred, 0)

print("Prediction summary before integer rounding:")
display(pd.Series(test_pred).describe())
print("Mean history scale:", test_long["scale"].mean())
print("Mean prediction  :", test_pred.mean())
print("Pred/scale ratio :", np.mean(test_pred / test_long["scale"].to_numpy()))

KeyError: "['horizon', 'age_rating', 'ticket_price_mean', 'day_tipe', 'holiday_tipe', 'genre_main', 'target_dow', 'target_day', 'target_month', 'target_dayofyear', 'target_weekofyear', 'target_is_weekend', 'target_is_month_start', 'target_is_month_end', 'target_is_holiday', 'hist_mean', 'hist_std', 'hist_min', 'hist_max', 'hist_median', 'hist_range', 'trend_13', 'trend_12', 'trend_23', 'ratio_21', 'ratio_31', 'ratio_32', 'log_scale', 'occ_mean', 'occ_std', 'occ_trend', 'show_mean', 'show_trend', 'tickets_per_show', 'tickets_per_occupancy', 'horizon_float', 'horizon_sq'] not in index"

## 13. Submission CSV — cek urutan ID dan format

File wajib memiliki kolom `id,total_ticket`. Kita merge berdasarkan `id` ke sample submission agar urutannya sama persis. Prediksi tiket dibulatkan menjadi bilangan bulat non-negatif.

In [18]:
pred_frame = test_long[["id"]].copy()
pred_frame["total_ticket"] = np.rint(test_pred).astype(np.int64)
pred_frame["total_ticket"] = pred_frame["total_ticket"].clip(lower=0)

submission = sample_sub[["id"]].merge(
    pred_frame, on="id", how="left", validate="one_to_one"
)

if submission["total_ticket"].isna().any():
    missing_ids = submission.loc[submission["total_ticket"].isna(), "id"].head(10).tolist()
    raise ValueError(f"Ada ID submission yang tidak mendapat prediksi. Contoh: {missing_ids}")

submission["total_ticket"] = submission["total_ticket"].astype(np.int64)

assert submission.shape[0] == sample_sub.shape[0]
assert submission["id"].equals(sample_sub["id"])
assert submission["total_ticket"].notna().all()
assert (submission["total_ticket"] >= 0).all()

output_path = os.path.join(BASE, "submission_tuning.csv")
submission.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", submission.shape)
display(submission.head())
display(submission["total_ticket"].describe())

NameError: name 'test_pred' is not defined

## 14. Feature importance dan simpan artefak eksperimen

Feature importance membantu memahami apa yang dipakai model. Ini bukan bukti kausal dan tidak otomatis berarti fitur terpenting akan meningkatkan private leaderboard.

In [19]:
importance = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_gain": final_model.booster_.feature_importance(importance_type="gain"),
    "importance_split": final_model.booster_.feature_importance(importance_type="split"),
}).sort_values("importance_gain", ascending=False)

display(importance.head(30))

results_path = os.path.join(BASE, "tuning_results.csv")
tuning_df.to_csv(results_path, index=False)

config_path = os.path.join(BASE, "best_config.json")
with open(config_path, "w") as f:
    json.dump({
        "seed": SEED,
        "n_trials": N_TRIALS,
        "validation_mase": float(best_score),
        "best_iteration": int(best_iteration or 0),
        "best_params": best_params,
        "validation_calibration_alpha": float(best_alpha),
        "use_calibration": bool(USE_CALIBRATION),
        "feature_columns": FEATURE_COLS,
        "categorical_features": cats_all,
    }, f, indent=2)

print("Saved tuning results:", results_path)
print("Saved best config:", config_path)

NameError: name 'final_model' is not defined

## 15. Checklist sebelum submit

- [ ] Validasi lokal dijalankan sampai selesai dan MASE tercatat.
- [ ] Periksa `horizon_report` untuk mengetahui apakah error memburuk pada D9–D10.
- [ ] Pastikan `submission_tuning.csv` memiliki tepat dua kolom: `id`, `total_ticket`.
- [ ] Pastikan jumlah baris dan urutan `id` sama dengan `sample_submission.csv`.
- [ ] Catat skor public leaderboard setelah submit, tetapi jangan menganggap public score sebagai jaminan private score.
- [ ] Jangan mengklaim MASE 0.33 sebelum hasil Kaggle benar-benar menunjukkan angka tersebut.